In [48]:
# Necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
from math import *

from matplotlib import animation
import sys
from IPython.display import HTML
from numpy import where

In [ ]:
# The following code is heavily inspired by the Math 714 GitHub j_fvolume code and code from: https://leifh.folk.ntnu.no/teaching/tkt4140/._main072.html (specifically 8.7.2). Constants such as seismic velocities and
# densities are from: https://web.ics.purdue.edu/~nowack/geos557/lecture9-dir/lecture9.htm

# Parameters
a = 5.79 # km/s
tmin = 0.0
tmax = 0.33
xmin = 0.0
xmax = 2.0
Nx = 100 # Spatial points
c = 0.9 # Courant number <=1 needed for stability

# Grid setup
x = np.linspace(xmin, xmax, Nx+1)
dx = float((xmax-xmin)/Nx)
dt = c/a*dx
Nt = int((tmax-tmin)/dt)
time = np.linspace(tmin, tmax, Nt)
#times = tmin + dt*np.arange(Nt + 1)

# Initial conditions
def f(x):
    f = np.zeros_like(x)
    f[np.where(x <= 0.1)] = 1.0 # Step function initial condition for wave propogation
    return f

# Lax-Friedrich Flux method
def lff(u):
    u[1: -1] = (u[:-2] + u[2:])/2.0 - dt*(a*u[2:] - a*u[:-2])/(2.0*dx)
    return u[1:-1]

# Solve code
solvers = [lff]

u_solutions = np.zeros((len(solvers) + 1, len(time), len(x)))

for k, solver in enumerate(solvers):
    u = f(x)
    un = np.zeros((len(time), len(x)))

    for i, t in enumerate(time[1:]):

        if k == 0:
            u_solutions[-1, i, :] = f(x - a*t) # Analytical solution for early timestep

        u_bc = interpolate.interp1d(x[-2:], u[-2:])

        u[1:-1] = solver(u[:]) # Numerical answer
        u[-1] = u_bc(x[-1] - a*dt) # Interpolate on char.

        un[i,:] = u[:]

    u_solutions[k,:,:] = un




# Animation

# First set up the figure, the axis, and the plot element we want to animate
fig = plt.figure()
ax = plt.axes(xlim=(xmin,xmax), ylim=(np.min(un), np.max(un)*1.1))

lines=[]     # list for plot lines for solvers and analytical solutions
legends=[]   # list for legends for solvers and analytical solutions

for solver in solvers:
    line, = ax.plot([], [], linewidth=2)
    lines.append(line)
    legends.append("Lax-Freidrich Solution")
    legends.append("Analytical")

line, = ax.plot([], [], linewidth=2) #add extra plot line for analytical solution
lines.append(line)

plt.xlabel('x-coordinate [-]')
plt.ylabel('Amplitude [-]')
plt.legend(legends, loc=3, frameon=False)
 
# initialization function: plot the background of each frame
def init():
    for line in lines:
        line.set_data([], [])
    return lines,

# animation function.  This is called sequentially
def animate(i):
    for k, line in enumerate(lines):
        line.set_data(x, un[i,:])
    return lines,

def animate_alt(i):
    for k, line in enumerate(lines):
        line.set_data(x, u_solutions[k,i,:])
    return lines,

 
# call the animator.  blit=True means only re-draw the parts that have changed.
anim = animation.FuncAnimation(fig, animate_alt, init_func=init, frames=Nt-1, interval=100, blit=False)
 
 
plt.close(fig)
HTML(anim.to_jshtml())